In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle
import numpy as np
import pandas as pd
import random
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import os
from torchmetrics.functional import structural_similarity_index_measure as ssim
#from run_trained_BM3DLUnet_model import run_BM3DLUnet_sig02, run_BM3DLUnet_sig03, run_BM3DLUnet_sig04
#from run_trained_BM3DLUnet_model import run_BM3DFullUnet_sig02, run_BM3DFullUnet_sig03, run_BM3DFullUnet_sig04
from run_trained_BM3DUnet_models_finalized import run_ResUnet_4CBCTDen, run_BM3DResUnet_4CBCTDen, run_SwinIR_4CBCTDen, run_SwinIR_4CBCTDen_bm3d, run_BM3DSwinIR_4CBCTDen, run_HARUnet_4CBCTDen, run_HARUnet_4CBCTDen_bm3d, run_BM3DHARUnet_4CBCTDen

from Full_Unet import ConvBlock, Heavy_UNet

from network_swinir import SwinIR
from HARUnet_model_ import HARU_net
from utils import psnr, batch_psnr

from torchmetrics.functional import structural_similarity_index_measure as ssim
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

from torchmetrics.image import MultiScaleStructuralSimilarityIndexMeasure as ms_ssim

import torch.nn.functional as F

import matplotlib.pyplot as plt

from bm3d import bm3d

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lpips_metric = LearnedPerceptualImagePatchSimilarity(net_type='vgg')  # or 'alex', 'squeeze'
compute_lpips = lpips_metric.to('cuda')
compute_msssim = ms_ssim(data_range=1.0).to(device)

c:\Users\au711969\AppData\Local\anaconda3\envs\KnTorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\au711969\AppData\Local\anaconda3\envs\KnTorch\Lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
c:\Users\au711969\AppData\Local\anaconda3\envs\KnTorch\Lib\site-packages\torchmetrics\functional\image\lpips.py:323: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main

In [2]:
def compute_gmsd(img1, img2, c=0.0026):
    """
    Gradient Magnitude Similarity Deviation (GMSD) metric.
    Args:
        img1, img2: tensors of shape (B, 1, H, W) or (B, 3, H, W), normalized to [0, 1]
        c: stability constant
    Returns:
        mean GMSD value across batch
    """
    if img1.ndim == 3:
        img1 = img1.unsqueeze(0)
        img2 = img2.unsqueeze(0)

    # Convert to grayscale if RGB
    if img1.shape[1] == 3:
        img1 = 0.299 * img1[:,0:1] + 0.587 * img1[:,1:2] + 0.114 * img1[:,2:3]
        img2 = 0.299 * img2[:,0:1] + 0.587 * img2[:,1:2] + 0.114 * img2[:,2:3]

    # Sobel filters
    sobel_x = torch.tensor([[1, 0, -1],
                            [2, 0, -2],
                            [1, 0, -1]], dtype=img1.dtype, device=img1.device).unsqueeze(0).unsqueeze(0)
    sobel_y = torch.tensor([[1, 2, 1],
                            [0, 0, 0],
                            [-1, -2, -1]], dtype=img1.dtype, device=img1.device).unsqueeze(0).unsqueeze(0)

    # Compute gradients
    grad_x1 = F.conv2d(img1, sobel_x, padding=1)
    grad_y1 = F.conv2d(img1, sobel_y, padding=1)
    grad_x2 = F.conv2d(img2, sobel_x, padding=1)
    grad_y2 = F.conv2d(img2, sobel_y, padding=1)

    gm1 = torch.sqrt(grad_x1**2 + grad_y1**2)
    gm2 = torch.sqrt(grad_x2**2 + grad_y2**2)

    # Gradient magnitude similarity map
    gms_map = (2 * gm1 * gm2 + c) / (gm1**2 + gm2**2 + c)

    # Standard deviation of the similarity map → GMSD
    gmsd_val = torch.std(gms_map, dim=[1, 2, 3])
    return gmsd_val.mean().item()


def compute_performance_metrics(ref, test):
    """Compute all metrics for a single pair."""
    # Ensure both are numpy arrays in [0,1]
    #ref = np.clip(ref, 0, 1)
    #test = np.clip(test, 0, 1)
    
    # Convert to torch tensor (C,H,W)
    #ref_t = torch.from_numpy(ref.transpose(2, 0, 1)).unsqueeze(0).to(device)
    #test_t = torch.from_numpy(test.transpose(2, 0, 1)).unsqueeze(0).to(device)

    psnr_val = batch_psnr(ref, test)

    ssim_val = ssim(ref, test)

    # GMSD
    gmsd_val = compute_gmsd(ref, test)

    return psnr_val, ssim_val, gmsd_val

In [3]:
# Load data
def load_data(pickle_file):
    with open(pickle_file, 'rb') as f:
        patches = pickle.load(f)
    return patches  # Expecting a NumPy array (N, 1, H, W)

class CBCTDataset(Dataset):
    def __init__(self, noisy_patches, target_patches):
        self.noisy = noisy_patches  # Keep as is
        self.target = target_patches  # Keep as is

    def __len__(self):
        return len(self.noisy)

    def __getitem__(self, idx):
        noisy_tensor = torch.tensor(self.noisy[idx], dtype=torch.float32)
        target_tensor = torch.tensor(self.target[idx], dtype=torch.float32)

        return noisy_tensor, target_tensor
    
pickled_test_inputs = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Test_noisyCBCT_patches.pkl"
pickled_test_targets = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Test_origCBCT_patches.pkl"
test_inputs = load_data(pickled_test_inputs)  # Shape: (N, 1, H, W)
test_targets = load_data(pickled_test_targets)  # Shape: (N, 1, H, W)
test_dataset = CBCTDataset(test_inputs, test_targets)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=True, num_workers=0,pin_memory=True)

In [4]:
#load models

model_dir = r"C:\Users\au711969\OneDrive - Aarhus universitet\Dentistry_Stuff\My Research projects\CBCT Denoising Project\BM3D_on_CBCT\Codes"

ResUnet_modelname = r"ResUnet_trainedon_noisytorawCBCTs_CadavarData_at_55epochs_newData_latest.pth"
#ResUnet_modelname = r"SupervisedLearning_FullUnet_trainedon_noisytorawCBCTs_CadavarData_at_23epochs_newData.pth"
model_ResUnet = torch.load(os.path.join(model_dir,ResUnet_modelname)).to(device)

#HARUnet_modelname = r"HARUnetv1_trainedon_noisytorawCBCTs_CadavarData_at_20epochs.pth"
HARUnet_modelname = r"HARUnetv2_11_trainedon_noisytorawCBCTs_CadavarData_at_42epochs_.pth"
model_HARUnet = torch.load(os.path.join(model_dir,HARUnet_modelname)).to(device)

SwinIR_modelname = r"SWINIR_trainedon_noisy2raw_CadavarData_at_39epochs_.pth"
model_SwinIR = torch.load(os.path.join(model_dir,SwinIR_modelname)).to(device)

# 2. Load the state dictionary
Uformer_modelname = r"Uformer_trainedon_noisy2raw_CadavarData_at_40epochs_.pth"
model_Uformer = torch.load(os.path.join(model_dir,Uformer_modelname)).to(device)

model_ResUnet.eval()
model_SwinIR.eval()
model_HARUnet.eval()
model_Uformer.eval()

C:\Users\au711969\AppData\Local\Temp\ipykernel_34140\2522611447.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_ResUnet = torch.load(os.path.join(model_dir,ResUnet

DataParallel(
  (module): Uformer(
    embed_dim=64, token_projection=linear, token_mlp=leff,win_size=8
    (pos_drop): Dropout(p=0.0, inplace=False)
    (input_proj): InputProj(
      (proj): Sequential(
        (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): LeakyReLU(negative_slope=0.01, inplace=True)
      )
    )
    (output_proj): OutputProj(
      (proj): Sequential(
        (0): Conv2d(128, 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      )
    )
    (encoderlayer_0): BasicUformerLayer(
      dim=64, input_resolution=(256, 256), depth=2
      (blocks): ModuleList(
        (0): LeWinTransformerBlock(
          dim=64, input_resolution=(256, 256), num_heads=1, win_size=8, shift_size=0, mlp_ratio=4.0,modulator=None
          (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (attn): WindowAttention(
            dim=64, win_size=(8, 8), num_heads=1
            (qkv): LinearProjection(
              (to_q): Linear(in

In [5]:

input_batchpsnr = 0.0
ResUnet_batchpsnr = 0.0
SwinIR_batchpsnr = 0.0
Uformer_batchpsnr = 0.0
HARUnet_batchpsnr = 0.0

input_batchssim = 0.0
ResUnet_batchssim = 0.0
Uformer_batchssim = 0.0
SwinIR_batchssim = 0.0
HARUnet_batchssim = 0.0

input_batchgmsd = 0.0
ResUnet_batchgmsd = 0.0
Uformer_batchgmsd = 0.0
SwinIR_batchgmsd = 0.0
HARUnet_batchgmsd = 0.0

In [6]:

for i, (test_inputs, test_targets) in enumerate(test_loader):

    test_inputs, test_targets = test_inputs.unsqueeze(1).to(device), test_targets.unsqueeze(1).to(device)
    
    in_psnr, in_ssim, in_gmsd = compute_performance_metrics(test_inputs, test_targets)
            
    with torch.no_grad():
        outputs_ResUnet = model_ResUnet(test_inputs)
        outputs_Uformer = model_Uformer(test_inputs)
        outputs_SwinIR = model_SwinIR(test_inputs)
        outputs_HARUnet = model_HARUnet(test_inputs)

    psnr_ResU, ssim_ResU, gmsd_ResU = compute_performance_metrics(outputs_ResUnet, test_targets)
    psnr_Uform, ssim_Uform, gmsd_Uform = compute_performance_metrics(outputs_Uformer, test_targets)
    psnr_SIR, ssim_SIR, gmsd_SIR = compute_performance_metrics(outputs_SwinIR, test_targets)
    psnr_HARU, ssim_HARU, gmsd_HARU = compute_performance_metrics(outputs_HARUnet, test_targets)
    
    input_batchpsnr += in_psnr
    ResUnet_batchpsnr += psnr_ResU
    Uformer_batchpsnr += psnr_Uform
    SwinIR_batchpsnr += psnr_SIR
    HARUnet_batchpsnr +=  psnr_HARU

    input_batchssim += in_ssim
    ResUnet_batchssim += ssim_ResU
    Uformer_batchssim += ssim_Uform
    SwinIR_batchssim += ssim_SIR
    HARUnet_batchssim +=  ssim_HARU

    input_batchgmsd += in_gmsd
    ResUnet_batchgmsd += gmsd_ResU
    Uformer_batchgmsd += gmsd_Uform
    SwinIR_batchgmsd += gmsd_SIR
    HARUnet_batchgmsd +=  gmsd_HARU


    print(f'Batch_{i}: PSNR: {psnr_ResU}/ {psnr_SIR}/ {psnr_HARU}, SSIM: {ssim_ResU}/ {ssim_SIR}/ {ssim_HARU}, GSMD: {gmsd_ResU}/ {gmsd_SIR}/ {gmsd_HARU}')

Test_inpsnr = input_batchpsnr / (len(test_loader)-1)
Test_ResUnet_psnr = ResUnet_batchpsnr / (len(test_loader)-1)
Test_Uformer_psnr = Uformer_batchpsnr / (len(test_loader)-1)
Test_SwinIR_psnr = SwinIR_batchpsnr / (len(test_loader)-1)
Test_HARUnet_psnr = HARUnet_batchpsnr / (len(test_loader)-1)

Test_inssim = input_batchssim / (len(test_loader)-1)
Test_ResUnet_ssim = ResUnet_batchssim / (len(test_loader)-1)
Test_Uformer_ssim = Uformer_batchssim / (len(test_loader)-1)
Test_SwinIR_ssim = SwinIR_batchssim / (len(test_loader)-1)
Test_HARUnet_ssim = HARUnet_batchssim / (len(test_loader)-1)


Test_ingmsd = input_batchgmsd / (len(test_loader)-1)
Test_ResUnet_gmsd = ResUnet_batchgmsd / (len(test_loader)-1)
Test_Uformer_gmsd = Uformer_batchgmsd / (len(test_loader)-1)
Test_SwinIR_gmsd = SwinIR_batchgmsd / (len(test_loader)-1)
Test_HARUnet_gmsd = HARUnet_batchgmsd / (len(test_loader)-1)

c:\Users\au711969\AppData\Local\anaconda3\envs\KnTorch\Lib\site-packages\torchmetrics\utilities\prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(


Batch_0: PSNR: 33.89231978149738/ 36.545042944328436/ 38.64497942163801, SSIM: 0.9590266942977905/ 0.9589531421661377/ 0.9592707753181458, GSMD: 0.11410130560398102/ 0.10617244988679886/ 0.09972509741783142
Batch_1: PSNR: 30.809664635005603/ 31.46771400959608/ 32.55793902191445, SSIM: 0.9143285155296326/ 0.919122040271759/ 0.9210205674171448, GSMD: 0.1534470170736313/ 0.14405405521392822/ 0.1392589509487152
Batch_2: PSNR: 35.21278522127098/ 36.26363414783515/ 37.44509260561142, SSIM: 0.9677063226699829/ 0.9665881991386414/ 0.9677281379699707, GSMD: 0.11808422207832336/ 0.10756438970565796/ 0.09839988499879837
Batch_3: PSNR: 31.87436487033725/ 33.3448649906595/ 39.20207903246765, SSIM: 0.9438684582710266/ 0.9440892934799194/ 0.9467737674713135, GSMD: 0.13153022527694702/ 0.12178865075111389/ 0.11177781224250793
Batch_4: PSNR: 36.03650767317321/ 36.79903480526965/ 37.17537283877667, SSIM: 0.9517459869384766/ 0.9518179893493652/ 0.9514168500900269, GSMD: 0.11895832419395447/ 0.11155346781

In [7]:
print(f'###############################  Testing Results  ##############################')

print(f'Method                    PSNR,       SSIM,     GMSD')
print(f'Input              >>    {Test_inpsnr:.4f},    {Test_inssim:.4f},    {Test_ingmsd:.4f}')

print(f'ResUnet            >>    {Test_ResUnet_psnr:.4f},    {Test_ResUnet_ssim:.4f},    {Test_ResUnet_gmsd:.4f}')
print(f'Uformer            >>    {Test_Uformer_psnr:.4f},    {Test_Uformer_ssim:.4f},    {Test_Uformer_gmsd:.4f}')
print(f'SwinIR             >>    {Test_SwinIR_psnr:.4f},    {Test_SwinIR_ssim:.4f},    {Test_SwinIR_gmsd:.4f}')
print(f'HARUnet            >>    {Test_HARUnet_psnr:.4f},    {Test_HARUnet_ssim:.4f},    {Test_HARUnet_gmsd:.4f}')

###############################  Testing Results  ##############################
Method                    PSNR,       SSIM,     GMSD
Input              >>    19.8446,    0.7450,    0.2214
ResUnet            >>    34.9505,    0.9503,    0.1237
Uformer            >>    36.1744,    0.9408,    0.1144
SwinIR             >>    36.0439,    0.9510,    0.1151
HARUnet            >>    37.4351,    0.9521,    0.1082
